# Chapter 14 Exercise 3 - Tune Matching with GTPSA

The original tutorial asks for the Tao command `set tune -mask *,~QF,~QD 0.08 0.14`. In this SciBmad version, the model lattice has two knobs: the QF family strength and the QD family strength. GTPSA is used to compute the local Jacobian

`d(Qx, Qy) / d(kqf, kqd)`

and a Newton step adjusts the two quadrupole strengths until the model tunes reach the target.

The PDF's phrase "difference between the model and design lattices" refers to plotting the optics difference around the ring, so the main plot below is beta beating. The tune difference is also shown as a small scalar bar plot because tunes are whole-ring quantities, not functions of `s`.

In [ ]:
function chapter14_common_file()
    candidates = [
        joinpath(@__DIR__, "..", "chapter14_common.jl"),
        joinpath(pwd(), "..", "chapter14_common.jl"),
        joinpath(pwd(), "lattices", "chapter_14", "chapter14_common.jl"),
        joinpath(pwd(), "Ring_Design_Tutorial_SciBmad", "lattices", "chapter_14", "chapter14_common.jl"),
        joinpath(dirname(pwd()), "lattices", "chapter_14", "chapter14_common.jl"),
    ]

    for candidate in candidates
        isfile(candidate) && return candidate
    end

    error("Could not find chapter14_common.jl. Start this notebook from the repository root or the exercises folder.")
end

include(chapter14_common_file())

using GTPSA
using LinearAlgebra

const TUNE_DESCRIPTOR = Descriptor(6, 2, 2, 1)
const DK_TUNE = params(TUNE_DESCRIPTOR)
const ZERO6 = [0, 0, 0, 0, 0, 0]

function tps_const(x)
    try
        return x[ZERO6]
    catch
        return x
    end
end

optics_table(tw) = hasproperty(tw, :table) ? tw.table : tw
parameter_gradient(x) = GTPSA.gradient(x, include_params=true)[7:end]

## Model and Design Lattices

`k = [kqf, kqd]` stores the two family strengths. When `knobs` is supplied, the QF and QD strengths are promoted to GTPSA parameter-dependent values, so the final tunes carry derivatives with respect to the two knobs.

In [ ]:
function sliced_drift(name, L, n)
    return [Drift(name=@sprintf("%s_%02d", name, i), L=L / n) for i in 1:n]
end

function sliced_quad(name, L, Kn1, n)
    return [Quadrupole(name=@sprintf("%s_%02d", name, i), L=L / n, Kn1=Kn1) for i in 1:n]
end

function build_tune_ring(k; knobs=nothing, n_quad=12, n_drift=24)
    strength(i) = isnothing(knobs) ? k[i] : k[i] + knobs[i]

    elements = vcat(
        sliced_quad("QF", 0.25, strength(1), n_quad),
        sliced_drift("D1", 1.00, n_drift),
        sliced_quad("QD", 0.25, strength(2), n_quad),
        sliced_drift("D2", 1.00, n_drift),
    )

    return Beamline(elements; species_ref=CH13_SPECIES, E_ref=CH13_E_REF)
end

function ring_tunes(k; knobs=nothing, descriptor=nothing, constants=true)
    ring = build_tune_ring(k; knobs=knobs)
    tw = isnothing(descriptor) ? twiss(ring) : twiss(ring, GTPSA_descriptor=descriptor)
    table = optics_table(tw)
    tunes = [table.phi_1[end], table.phi_2[end]]
    return constants ? tps_const.(tunes) : tunes
end

## GTPSA Jacobian

The residual is

`r(k) = Q(k) - Q_target`.

With GTPSA knobs attached to the quadrupole strengths, each residual component carries its parameter derivatives. Reading those derivatives gives the response matrix used in the Newton update.

In [ ]:
function tune_residual(k, target)
    return ring_tunes(k) - target
end

function tune_residual_with_knobs(k, target)
    return ring_tunes(k; knobs=DK_TUNE, descriptor=TUNE_DESCRIPTOR, constants=false) .- target
end

function tune_jacobian(k, target)
    residual = tune_residual_with_knobs(k, target)
    return vcat((parameter_gradient(r)' for r in residual)...)
end

function match_tunes(k0, target; max_iter=10, tolerance=1e-9)
    k = copy(k0)

    for iteration in 1:max_iter
        r = tune_residual(k, target)
        tunes = r + target
        @printf(
            "iteration %2d: Qx = %.9f, Qy = %.9f, residual norm = %.3e\n",
            iteration,
            tunes[1],
            tunes[2],
            norm(r),
        )

        norm(r) < tolerance && break

        J = tune_jacobian(k, target)
        step = -(J \ r)
        step_norm = norm(step)
        step_norm > 0.25 && (step .*= 0.25 / step_norm)
        k .+= step
    end

    return k
end

## Plot Model - Design Difference

The beta beating is plotted as

`100 * (beta_model - beta_design) / beta_design`.

The second panel shows the scalar tune shift from the design lattice to the tuned model lattice.

In [ ]:
function plot_model_design_difference(k_design, k_model)
    design_table = optics_table(twiss(build_tune_ring(k_design)))
    model_table = optics_table(twiss(build_tune_ring(k_model)))

    s = tps_const.(model_table.s)
    beta_x_design = tps_const.(design_table.beta_1)
    beta_y_design = tps_const.(design_table.beta_2)
    beta_x_model = tps_const.(model_table.beta_1)
    beta_y_model = tps_const.(model_table.beta_2)

    beating_x = 100 .* (beta_x_model .- beta_x_design) ./ beta_x_design
    beating_y = 100 .* (beta_y_model .- beta_y_design) ./ beta_y_design
    tune_difference = ring_tunes(k_model) - ring_tunes(k_design)

    fig = Figure(size=(800, 620))
    ax1 = Axis(
        fig[1, 1],
        xlabel="s [m]",
        ylabel="beta beating [%]",
        title="beta(model) - beta(design)",
    )
    lines!(ax1, s, beating_x; color=:royalblue3, linewidth=2, label="beta_1")
    lines!(ax1, s, beating_y; color=:darkorange2, linewidth=2, label="beta_2")
    hlines!(ax1, [0.0]; color=:gray45, linestyle=:dash)
    axislegend(ax1; position=:rt)

    ax2 = Axis(
        fig[2, 1],
        ylabel="tune difference",
        title="Tune(model) - tune(design)",
        xticks=([1, 2], ["Qx", "Qy"]),
    )
    barplot!(ax2, [1, 2], tune_difference; color=[:royalblue3, :darkorange2])
    hlines!(ax2, [0.0]; color=:gray45, linestyle=:dash)

    return fig
end

In [ ]:
K_DESIGN = [0.30, -0.30]
TARGET_TUNES = [0.08, 0.14]

design_tunes = ring_tunes(K_DESIGN)
println("Design strengths: ", K_DESIGN)
println("Design tunes:     ", design_tunes)

K_MODEL = match_tunes(K_DESIGN, TARGET_TUNES)
model_tunes = ring_tunes(K_MODEL)

println("\nMatched strengths: ", K_MODEL)
println("Matched tunes:     ", model_tunes)
println("Tune residual:     ", model_tunes - TARGET_TUNES)
println("Model - design:    ", model_tunes - design_tunes)

In [ ]:
exercise3_figure = plot_model_design_difference(K_DESIGN, K_MODEL)
display(exercise3_figure)